# pulseboard-desktop — analysis notebook

Loads a result folder from `C:\NMDiag\<timestamp>-<tag>\` and produces a quick visual summary of:
- per-target loss + RTT over time
- anycast DNS-vs-CDN divergence (the money pivot)
- speedtest history (capacity at each point during the run)
- Wi-Fi RSSI + link rate
- worst-loss windows

**This is a stub.** Sponsor-funded improvements include: PDF report export, automated bad-window detection with annotated charts, side-by-side A/B comparison of two result folders. See [`README.md`](../README.md#sponsor-this-project).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Edit this to point at your result folder:
RESULT_DIR = Path(r'C:\NMDiag\20260425-150000-wifi-A')
assert RESULT_DIR.exists(), f'Result folder not found: {RESULT_DIR}'
print(f'Loading: {RESULT_DIR}')

## 1. Preflight summary
Read the bottom-line counts and any FAIL rows.

In [ ]:
preflight_path = RESULT_DIR / 'PREFLIGHT.txt'
with preflight_path.open(encoding='utf-8-sig') as f:
    text = f.read()
summary_line = [l for l in text.splitlines() if l.startswith('PREFLIGHT SUMMARY')]
print(summary_line[0] if summary_line else '(no summary found)')
fail_rows = [l for l in text.splitlines() if l.startswith('[FAIL]')]
if fail_rows:
    print('\nFAIL rows:')
    for r in fail_rows: print(f'  {r}')

## 2. Per-target ping loss + RTT over time

In [ ]:
pings = pd.read_csv(RESULT_DIR / 'pings.csv', encoding='utf-8-sig')
pings['timestamp'] = pd.to_datetime(pings['timestamp'])
pings.set_index('timestamp', inplace=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for tgt, grp in pings.groupby('target'):
    ax1.plot(grp.index, grp['loss_pct'], label=tgt, alpha=0.7)
    ax2.plot(grp.index, grp['avg_ms'],   label=tgt, alpha=0.7)
ax1.set_ylabel('Loss %'); ax1.legend(loc='upper right'); ax1.grid(True, alpha=0.3)
ax2.set_ylabel('Avg RTT (ms)'); ax2.legend(loc='upper right'); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Anycast DNS-vs-CDN divergence
If the ISP fast-lanes DNS-anycast and routes CDN-anycast through a slower path, the cross-tier pairs (`cloudflare_dns_vs_cdn`, `google_dns_vs_api`) will show a consistent gap.

In [ ]:
anycast = pd.read_csv(RESULT_DIR / 'anycast.csv', encoding='utf-8-sig')
anycast['timestamp'] = pd.to_datetime(anycast['timestamp'])
anycast.set_index('timestamp', inplace=True)

fig, ax = plt.subplots(figsize=(14, 5))
for pair, grp in anycast.groupby('pair'):
    ax.plot(grp.index, grp['divergence_ms'], label=pair, alpha=0.8)
ax.axhline(20, color='orange', linestyle='--', alpha=0.5, label='20ms (notable)')
ax.axhline(50, color='red', linestyle='--', alpha=0.5, label='50ms (significant)')
ax.set_ylabel('Divergence (ms)'); ax.legend(loc='upper right'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nMean divergence per pair:')
print(anycast.groupby('pair')['divergence_ms'].agg(['mean','min','max']))

## 4. Speedtest history — actual capacity at each hour

In [ ]:
import json
speed = []
for jf in sorted((RESULT_DIR / 'speedtest').glob('*.json')):
    try:
        # Speedtest CLI writes UTF-16 via PowerShell stdout redirect — try both encodings
        for enc in ('utf-16', 'utf-8-sig', 'utf-8'):
            try:
                with jf.open(encoding=enc) as f:
                    d = json.loads(f.read())
                break
            except Exception:
                continue
        speed.append({
            'timestamp': pd.to_datetime(d.get('timestamp')),
            'down_mbps': d.get('download', {}).get('bandwidth', 0) * 8 / 1e6,
            'up_mbps':   d.get('upload',   {}).get('bandwidth', 0) * 8 / 1e6,
            'ping_ms':   d.get('ping', {}).get('latency', 0),
            'jitter_ms': d.get('ping', {}).get('jitter', 0),
            'server':    d.get('server', {}).get('name', '?'),
        })
    except Exception as e:
        print(f'skipped {jf.name}: {e}')
speed_df = pd.DataFrame(speed)
print(speed_df.to_string(index=False))

## 5. Worst-loss windows
The 10 minutes during the run with the most cumulative loss across all non-control targets — the windows you'd attach to a support ticket.

In [ ]:
loss_per_minute = pings.groupby([pd.Grouper(freq='1min'), 'target'])['loss_pct'].mean().unstack()
loss_per_minute['total'] = loss_per_minute.sum(axis=1)
worst = loss_per_minute.sort_values('total', ascending=False).head(10)
print('Worst 10 minutes by total loss across targets:')
print(worst.to_string())

## What's next
Sponsor-funded improvements:
- Auto-detect bad-call windows from a separate `bad-call-log.csv` you maintain, overlay on these charts
- Per-hop pathping loss visualisation (parses `mtr/*.txt`)
- pcap analysis: TCP retransmit rate, TLS handshake latency, DNS query timing
- A/B compare two result folders side-by-side